# LLM Skill Analysis

In [ ]:
import json
import pandas as pd
from pathlib import Path
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv() 

All csv files being written to 'analysis_gpt' folder

Uses gpt model through OpenAI API token


In [ ]:
SUBSET_PATH = "llm_analysis_subset.csv"
BASELINES_PATH = "full_alexa_skills_with_baselines.csv"
DIRECTORY = Path("analysis_gpt_old")

client = OpenAI()

MODEL = "gpt-4o-mini"

## 1. Load Subset

In [6]:
# llm_analysis_subset.csv
subset = pd.read_csv(SUBSET_PATH)
print("Subset:", subset.shape)

# Alexa dataset
baselines = pd.read_csv(BASELINES_PATH)
print("Baselines:", baselines.shape)

subset[["Skill Name", "Invocation Name", "Category", "baseline_risk_score", "selection_reasons"]].head()

Subset: (1244, 62)
Baselines: (42202, 61)


,Skill Name,Invocation Name,Category,baseline_risk_score,selection_reasons
0,Cricket Facts,cricket facts,Business & Finance,0.834719,baseline_high_risk;high_genericity_ambiguity
1,bitcoin facts,crypto facts,Business & Finance,0.743463,baseline_high_risk;high_genericity_ambiguity
2,Chore chart,chore chart,Business & Finance,0.656681,baseline_high_risk
3,AI NVR Integration Test,my assistant,Business & Finance,0.643772,baseline_high_risk
4,Cryptocurrency,crypto currency,Business & Finance,0.639859,baseline_high_risk


## 2. Prompt 

In [7]:
def build_prompt(row):
    return f"""
You are analyzing potential security and usability risks of Alexa skill invocation names.

Evaluate the following skill and return a structured JSON response.

Skill Information:
- Skill Name: {row['Skill Name']}
- Invocation Name: {row['Invocation Name']}
- Category: {row['Category']}
- Description: {row['Description']}
- Average Rating: {row['Average Rating']}
- Number of Ratings: {row['Number of Rating']}

Context:
- Invocation names are spoken aloud by users.
- Users may mispronounce, abbreviate, or paraphrase names.
- Attackers may create similar sounding or semantically similar invocation names.

Assign an integer score from 0 to 5 for each dimension.

Use the FULL scale:
0 = no meaningful risk
1 = very low risk
2 = low risk
3 = moderate risk
4 = high risk
5 = very high risk

Do NOT default to middle scores.
Use 0 or 5 when appropriate.

Scoring dimensions:

1. Genericity Risk
0 = highly distinctive and unique
1 = mostly distinctive
2 = somewhat common
3 = moderately generic
4 = highly generic/common
5 = extremely generic, broad everyday phrase

2. Ambiguity Risk
0 = only one clear meaning
1 = very little ambiguity
2 = minor ambiguity
3 = moderate ambiguity
4 = multiple plausible meanings
5 = highly ambiguous, many interpretations

3. Semantic Confusion Risk
0 = unlikely to be confused with other skills
1 = minimal confusion potential
2 = slight confusion potential
3 = moderate confusion potential
4 = high confusion potential
5 = extremely easy to confuse with other skills

4. Squatting Risk
0 = difficult to imitate or exploit
1 = very low squatting potential
2 = low squatting potential
3 = moderate squatting potential
4 = high squatting potential
5 = extremely easy to imitate with malicious variants

5. Overall Risk
0 = negligible overall risk
1 = very low overall risk
2 = low overall risk
3 = moderate overall risk
4 = high overall risk
5 = severe overall risk

Return ONLY valid JSON in this format:

{{
    "genericity_risk": int,
    "ambiguity_risk": int,
    "semantic_confusion_risk": int,
    "squatting_risk": int,
    "overall_risk": int,
    "reasoning": "concise explanation (2-4 sentences)"
}}
"""

## 3. Load Functions

In [ ]:
SCORE_FIELDS = ["genericity_risk", "ambiguity_risk", "semantic_confusion_risk",
                "squatting_risk", "overall_risk"]

def run_model(prompt):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return resp.choices[0].message.content.strip()


def parse_response(resp):
    print("parsing response...")
    base = {}

    for field in SCORE_FIELDS:
        base["llm_" + field] = None


    base["llm_reasoning"] = None
    if resp is None:
        return base

    start = resp.find("{")
    end = resp.rfind("}")

    if start == -1 or end == -1:
        return base

    try:
        json_str = resp[start:end + 1].replace("\\'", "'")
        obj = json.loads(json_str)


        for field in SCORE_FIELDS:
            if field in obj:
                base["llm_" + field] = int(obj[field])
        base["llm_reasoning"] = str(obj.get("reasoning", ""))
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    return base

## 4. Run Analysis

In [11]:
new_rows = []

for index, row in subset.iterrows():
    raw_response = run_model(build_prompt(row))
    parsed_data  = parse_response(raw_response)

    combined_row = row.to_dict()
    combined_row.update(parsed_data)
    new_rows.append(combined_row)

    print(index + 1, "rows completed")

results_df = pd.DataFrame(new_rows)
print("Results:", results_df.shape)

parsing response...
1 rows completed
parsing response...
2 rows completed
parsing response...
3 rows completed
parsing response...
4 rows completed
parsing response...
5 rows completed
parsing response...
6 rows completed
parsing response...
7 rows completed
parsing response...
8 rows completed
parsing response...
9 rows completed
parsing response...
10 rows completed
parsing response...
11 rows completed
parsing response...
12 rows completed
parsing response...
13 rows completed
parsing response...
14 rows completed
parsing response...
15 rows completed
parsing response...
16 rows completed
parsing response...
17 rows completed
parsing response...
18 rows completed
parsing response...
19 rows completed
parsing response...
20 rows completed
parsing response...
21 rows completed
parsing response...
22 rows completed
parsing response...
23 rows completed
parsing response...
24 rows completed
parsing response...
25 rows completed
parsing response...
26 rows completed
parsing response...
2

In [12]:
results_df.to_csv("llm_results.csv")
print("saved")

saved


## 5. Score Distributions

In [13]:
llm_columns = []

for field in SCORE_FIELDS:
    column_name = "llm_"+str(field)
    llm_columns.append(column_name)

new_data = results_df.dropna(subset=llm_columns).copy()
description = new_data[llm_columns].describe().round(2)

## 6. Compare LLM vs Baseline

In [14]:
new_data["llm_composite"] = new_data["llm_overall_risk"]

corr = new_data[["baseline_risk_score", "llm_composite"]].corr().iloc[0, 1]
print(f"baseline_risk_score vs llm_overall_risk: {corr:.5f}")

pop_quartiles = baselines["baseline_risk_score"].quantile([0.25, 0.5, 0.75])
new_data["baseline_population_tier"] = pd.cut( new_data["baseline_risk_score"],
    bins=[-np.inf, pop_quartiles[0.25], pop_quartiles[0.5], pop_quartiles[0.75], np.inf],
    labels=["Q1 low", "Q2", "Q3", "Q4 high"],
)

new_data["llm_quartile"] = pd.qcut(new_data["llm_composite"], q=4, labels=False, duplicates="drop")

print()
print("LLM score distribution:")
print(new_data["llm_composite"].value_counts().sort_index())


print()
print("Baseline vs LLM overall risk:")
print(pd.crosstab(new_data["baseline_population_tier"], new_data["llm_composite"]))

comparison_cols = [
    "Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "baseline_population_tier",
    "llm_genericity_risk", "llm_ambiguity_risk",
    "llm_semantic_confusion_risk", "llm_squatting_risk",
    "llm_overall_risk", "llm_reasoning",
]
new_data[comparison_cols].to_csv("llm_vs_baseline.csv", index=False)
print()
print("Wrote")

baseline_risk_score vs llm_overall_risk: 0.38865

LLM score distribution:
llm_composite
1      4
2    193
3    374
4    654
5     19
Name: count, dtype: int64

Baseline vs LLM overall risk:
llm_composite             1   2    3    4  5
baseline_population_tier                    
Q1 low                    3  88   91  104  4
Q2                        0  50   74   71  3
Q3                        1  28   86   70  4
Q4 high                   0  27  123  409  8

Wrote


## 7. False positives vs False Negatives

In [15]:
fp = new_data[
    (new_data["baseline_risk_score"] >= new_data["baseline_risk_score"].quantile(0.75)) &
    (new_data["llm_composite"] <=new_data["llm_composite"].quantile(0.25))
].sort_values("baseline_risk_score", ascending=False)

print(f"False Positives: {len(fp)} skills")
fp[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

False Positives: 62 skills


,Skill Name,Invocation Name,Category,baseline_risk_score,llm_composite,llm_reasoning
293,Cricket Facts,cricket facts,Game and Trivia,0.834719,3,The invocation name 'cricket facts' is somewha...
608,cricket,cricket facts,Movie and TV,0.834719,3,The invocation name 'cricket facts' is somewha...
989,Cricket Facts,cricket facts,Sports,0.834719,3,The invocation name 'cricket facts' is somewha...
990,Cricket Facts,cricket facts,Sports,0.834719,3,The invocation name 'cricket facts' is somewha...
459,Science Facts,science facts,Kids,0.819664,3,The invocation name 'science facts' is quite g...
241,Food Facts,food facts,Food and Drink,0.816562,3,The invocation name 'food facts' is somewhat g...
754,food facts,food facts,Novelty and Humor,0.816562,3,The invocation name 'food facts' is somewhat g...
753,Fun Food Facts Skill,food facts,Novelty and Humor,0.816562,3,The invocation name 'food facts' is quite gene...
305,Food facts,food facts,Game and Trivia,0.816562,3,The invocation name 'food facts' is quite gene...
757,Animal Facts,animal facts,Novelty and Humor,0.785496,3,The invocation name 'animal facts' is quite ge...


In [16]:
fn = new_data[
    (new_data["baseline_risk_score"] <= new_data["baseline_risk_score"].quantile(0.25)) &
    (new_data["llm_composite"] >=new_data["llm_composite"].quantile(0.75))
].sort_values("llm_composite", ascending=False)

print(f"False Negatives: {len(fn)} skills")
fn[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

disagreement_cols = ["Skill Name", "Invocation Name", "Category",
                     "baseline_risk_score", "llm_composite", "llm_reasoning"]

fp[disagreement_cols].to_csv("false_positives.csv", index=False)
fn[disagreement_cols].to_csv("false_negatives.csv", index=False)

print()
print(f"Wrote analysis/false_positives.csv {len(fp)} rows")
print(f"Wrote analysis/false_negatives.csv {len(fn)} rows")

False Negatives: 113 skills

Wrote analysis/false_positives.csv 62 rows
Wrote analysis/false_negatives.csv 113 rows


In [18]:
fp_by_category = (
    fp.groupby("Category")
      .agg(
          fp_count=("Skill Name", "count"),
          avg_baseline=("baseline_risk_score", "mean"),
          avg_llm=("llm_composite", "mean")
      )
      .sort_values("fp_count", ascending=False)
)

fn_by_category = (
    fn.groupby("Category")
      .agg(
          fn_count=("Skill Name", "count"),
          avg_baseline=("baseline_risk_score", "mean"),
          avg_llm=("llm_composite", "mean")
      )
      .sort_values("fn_count", ascending=False)
)

category_sizes = new_data.groupby("Category").size().rename("total_skills")

In [19]:
fp_rate = fp_by_category.join(category_sizes)
fp_rate["fp_rate"] = fp_rate["fp_count"] / fp_rate["total_skills"]
fp_rate.sort_values("fp_rate", ascending=False)

,fp_count,avg_baseline,avg_llm,total_skills,fp_rate
Category,,,,,
Novelty and Humor,10,0.773919,2.800000,56,0.178571
Movie and TV,7,0.746635,2.857143,46,0.152174
Food and Drink,7,0.715966,2.714286,46,0.152174
Kids,6,0.769533,2.833333,52,0.115385
Lifestyle,5,0.719032,3.000000,48,0.104167
productivity,5,0.716307,3.000000,50,0.100000
News,4,0.721464,3.000000,45,0.088889
Sports,4,0.773310,2.750000,46,0.086957
Music and Audio,3,0.668548,3.000000,54,0.055556


In [20]:
fn_rate = fn_by_category.join(category_sizes)
fn_rate["fn_rate"] = fn_rate["fn_count"] / fn_rate["total_skills"]
fn_rate.sort_values("fn_rate", ascending=False)

,fn_count,avg_baseline,avg_llm,total_skills,fn_rate
Category,,,,,
Social,8,0.209959,4.000000,50,0.160000
Food and Drink,7,0.229615,4.000000,46,0.152174
Communication,10,0.228321,4.000000,67,0.149254
Lifestyle,7,0.245035,4.142857,48,0.145833
productivity,7,0.237424,4.142857,50,0.140000
weather,7,0.246229,4.000000,52,0.134615
Game and Trivia,7,0.209786,4.142857,56,0.125000
Kids,6,0.228110,4.166667,52,0.115385
Home services,6,0.238830,4.000000,53,0.113208


## 8. Category Summary

In [17]:
summary = (

    new_data.groupby("Category").agg(
        skill_count = ("Skill Name","count"),
        mean_baseline = ("baseline_risk_score","mean"),
        mean_llm_genericity_risk = ("llm_genericity_risk","mean"),
        mean_llm_ambiguity_risk = ("llm_ambiguity_risk","mean"),
        mean_llm_semantic_confusion = ("llm_semantic_confusion_risk","mean"),
        mean_llm_squatting_risk = ("llm_squatting_risk","mean"),
        mean_llm_overall_risk = ("llm_overall_risk","mean"),
        mean_llm_composite = ("llm_composite","mean"),
    ).sort_values("mean_llm_overall_risk", ascending=False).reset_index()
)
summary.to_csv("llm_category_summary.csv", index=False)
summary.round(2)

,Category,skill_count,mean_baseline,mean_llm_genericity_risk,mean_llm_ambiguity_risk,mean_llm_semantic_confusion,mean_llm_squatting_risk,mean_llm_overall_risk,mean_llm_composite
0,productivity,50,0.51,3.96,2.28,3.04,3.62,3.74,3.74
1,Social,50,0.49,3.88,2.26,2.94,3.64,3.74,3.74
2,Education & Reference,56,0.57,3.93,2.09,2.95,3.68,3.70,3.70
3,Kids,52,0.57,3.81,2.06,2.94,3.67,3.65,3.65
4,Game and Trivia,56,0.56,3.70,1.98,2.82,3.64,3.62,3.62
5,Communication,67,0.41,3.81,2.18,2.94,3.54,3.57,3.57
6,Home services,53,0.41,3.62,2.26,2.91,3.49,3.55,3.55
7,Lifestyle,48,0.50,3.81,1.88,2.83,3.56,3.54,3.54
8,Business & Finance,68,0.42,3.46,2.09,2.81,3.44,3.40,3.40
9,Travel and Transportation,50,0.42,3.58,2.06,2.78,3.44,3.38,3.38
